In [15]:
import sys
sys.path.insert(0, '.')
from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries

In [37]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
# Define the system prompt
sys_msg = SystemMessage(role ="system", content="You are a helpful assistant that can answer only math questions.")
human_msg = HumanMessage(role="user", content="What is 2 + 2?") 
# Create a prompt template
prompt_template = PromptTemplate.from_template("{sys_msg}\n{human_msg}")
# Format the prompt with the system and human messages
formatted_prompt = prompt_template.format(sys_msg=sys_msg.content, human_msg=human_msg.content)

# Get the LLM
llm = get_llm() 

# Run the LLM with the formatted prompt
response = llm.invoke(formatted_prompt)

print(response)
print(response.content)

content='2 + 2 equals 4.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 27, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b5e2cd5e9e', 'id': 'chatcmpl-Ds41RanFCL9EnQqkfNiOViJK95MSE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019eda37-b5f2-7111-8da9-e2c7669b51cf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 8, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
2 + 2 equals 4.


In [ ]:
PromptTemplate.from

In [ ]:


from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm()
print(f'LLM ready | Tools available: {[t.name for t in TOOLS]}')

LLM ready | Tools available: ['calculate', 'search_docs', 'get_weather']


### Step 1 — bind_tools: Register Tools With the LLM
`bind_tools` sends tool descriptions (name, docstring, parameters) to the LLM
so it knows what tools are available. The LLM does NOT execute them — it just knows about them.

In [3]:
llm_with_tools = llm.bind_tools(TOOLS)

# What does the LLM output when it wants to use a tool?
response = llm_with_tools.invoke('What is 144 divided by 12?')

print(f'response.content    = "{response.content}"')      # empty — LLM chose a tool
print(f'response.tool_calls = {response.tool_calls}')    # LLM output: PLAN to call calculate
print()
print('Notice: the LLM did NOT compute 12. It output a tool call instruction.')
print('YOUR code must now read this and actually run calculate("144/12").')

response.content    = ""
response.tool_calls = [{'name': 'calculate', 'args': {'expression': '144 / 12'}, 'id': 'call_yMeT8UG4ukPe8LwvA14FjiSL', 'type': 'tool_call'}]

Notice: the LLM did NOT compute 12. It output a tool call instruction.
YOUR code must now read this and actually run calculate("144/12").


### Step 2 — YOU Execute the Tool (Manual Loop)
Read the tool call, run the function, wrap the result in `ToolMessage`, send back to LLM.
You are writing what an agent framework would do automatically.

In [4]:
tool_map = {t.name: t for t in TOOLS}

query    = 'What is 144 divided by 12?'
messages = [HumanMessage(content=query)]

# Round 1: LLM decides which tool to call
response = llm_with_tools.invoke(messages)
messages.append(response)
print(f'Step 1 — LLM says: call {response.tool_calls[0]["name"]}({response.tool_calls[0]["args"]})')

# Round 2: YOUR code runs the tool
for call in response.tool_calls:
    result = tool_map[call['name']].invoke(call['args'])
    print(f'Step 2 — YOU run:  {call["name"]}({call["args"]}) → {result}')
    messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

# Round 3: LLM sees tool result and forms final answer
final = llm_with_tools.invoke(messages)
print(f'Step 3 — LLM says: {final.content}')
print()
print('You manually did 3 steps. An agent framework does this automatically.')

Step 1 — LLM says: call calculate({'expression': '144 / 12'})
Step 2 — YOU run:  calculate({'expression': '144 / 12'}) → 12.0
Step 3 — LLM says: 144 divided by 12 is 12.

You manually did 3 steps. An agent framework does this automatically.


### Step 3 — Direct Answer (No Tool Called)
When the LLM can answer from training data, it skips tools and answers directly.
In this case `response.tool_calls` is empty.

In [5]:
r = llm_with_tools.invoke('What does LCEL stand for in LangChain?')
print(f'tool_calls: {r.tool_calls}')   # empty — LLM answered directly
print(f'content:    {r.content}')

tool_calls: [{'name': 'search_docs', 'args': {'query': 'LCEL LangChain'}, 'id': 'call_kERs8DsNvvZgDirJvcnwNWQn', 'type': 'tool_call'}]
content:    


### Step 4 — Multi-Tool Case
The LLM can request multiple tools in one response. Your code must loop through all of them.

In [ ]:
messages = [HumanMessage(content='Search docs for RAG and also calculate 25 multiplied by 4')]
r = llm_with_tools.invoke(messages)
messages.append(r)

print(f'LLM requested {len(r.tool_calls)} tool(s):')
for call in r.tool_calls:
    result = tool_map[call['name']].invoke(call['args'])
    print(f'  {call["name"]}({call["args"]}) → {str(result)[:80]}')
    messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

final = llm_with_tools.invoke(messages)
print(f'\nFinal answer: {final.content}')

### Full Function — Manual LLM + Tools Loop
This is the complete pattern you'd write without an agent framework.

In [ ]:
def llm_with_tools_call(query: str) -> str:
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content

### Test All Queries

In [ ]:
run_queries(llm_with_tools_call, label='Foundation — LLM + Tool Binding (Manual)')

### Summary — What's Missing Here?

```
Problem 1: YOU write the loop — if the LLM needs to call 5 tools in sequence, you
           need to handle every iteration manually.

Problem 2: No reasoning trace — you see what the LLM decided but not WHY.

Problem 3: No autonomous decision-making — the LLM cannot decide 'I need one more
           tool call' after seeing a result unless you code that logic.

→ Notebook 2 (ReAct Agent): agent writes its reasoning before every action.
→ Notebook 3 (Tool Calling Agent): create_agent manages the loop so you don't have to.
```